# 10 — Consolidação dos Gráficos Finais

## Objetivo

Este reúne os gráficos mais relevantes da modelagem e da aplicação estratégica em uma única pasta, com identidade visual padronizada.

## Princípios

- não treina nem altera modelos;
- não recalcula o limiar de decisão;
- não modifica as bases Gold;
- lê somente artefatos produzidos pelos notebooks 08 e 09;
- grava exclusivamente em `tech_challenge_fase3/images/finais`;
- mantém títulos, unidades, fontes e nomes de arquivos consistentes.


## 1. Configuração visual e caminhos

A pasta `images/finais` funcionará como a fonte oficial das figuras da entrega. O manifesto será salvo em `reports/consolidacao_visual` para permitir auditoria dos arquivos gerados.


In [0]:
# Objetivo: configurar bibliotecas, caminhos e padrão visual.

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    average_precision_score,
    roc_auc_score
)

FASE3_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase3"
)

RESULTADOS_MODELAGEM = f"{FASE3_ROOT}/modelagem/resultados"
RELATORIOS_MODELAGEM = f"{FASE3_ROOT}/reports/modelagem"
RELATORIOS_APLICACAO = f"{FASE3_ROOT}/reports/aplicacao_estrategica"
IMAGENS_APLICACAO = f"{FASE3_ROOT}/images/aplicacao_estrategica"
IMAGENS_FINAIS = f"{FASE3_ROOT}/images/finais"
RELATORIO_VISUAL = f"{FASE3_ROOT}/reports/consolidacao_visual"

dbutils.fs.mkdirs(IMAGENS_FINAIS)
dbutils.fs.mkdirs(RELATORIO_VISUAL)

CORES = {
    "azul": "#2E75B6",
    "vermelho": "#C44E52",
    "verde": "#55A868",
    "laranja": "#DD8452",
    "cinza": "#7F7F7F",
    "claro": "#EAF2F8"
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 180,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "legend.frameon": False
})

arquivos_gerados = []


def salvar_figura(figura, nome, titulo_executivo, origem):
    caminho = f"{IMAGENS_FINAIS}/{nome}"
    figura.savefig(
        caminho,
        dpi=180,
        bbox_inches="tight",
        facecolor="white"
    )
    arquivos_gerados.append({
        "arquivo": nome,
        "titulo": titulo_executivo,
        "origem": origem,
        "caminho": caminho
    })
    plt.show()
    plt.close(figura)


print("Pasta oficial de imagens:", IMAGENS_FINAIS)


## 2. Carregamento e validação dos relatórios

Todos os arquivos abaixo já foram produzidos pelos notebooks anteriores. A execução será interrompida se algum insumo obrigatório estiver ausente, evitando gráficos parciais ou silenciosamente incorretos.


In [0]:
# Objetivo: carregar os relatórios consolidados dos notebooks 08 e 09.

def ler_csv(caminho):
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Artefato obrigatório não encontrado: {caminho}")
    return pd.read_csv(caminho, sep=";", encoding="utf-8-sig")


intervalos = ler_csv(
    f"{RELATORIOS_MODELAGEM}/intervalos_confianca.csv"
)
metricas_regiao = ler_csv(
    f"{RELATORIOS_MODELAGEM}/metricas_por_regiao.csv"
)
importancias = ler_csv(
    f"{RELATORIOS_MODELAGEM}/importancia_permutacao.csv"
)
limiares = ler_csv(
    f"{RESULTADOS_MODELAGEM}/analise_limiares_otimizado.csv"
)
ranking_risco = ler_csv(
    f"{RELATORIOS_APLICACAO}/ranking_municipal_risco.csv"
)
risco_meta = ler_csv(
    f"{RELATORIOS_APLICACAO}/municipios_risco_meta.csv"
)
avaliacao_clusters = ler_csv(
    f"{RELATORIOS_APLICACAO}/avaliacao_k_clusters.csv"
)
clusters_regiao = ler_csv(
    f"{RELATORIOS_APLICACAO}/distribuicao_clusters_regiao.csv"
)

caminho_previsoes = (
    f"{RELATORIOS_MODELAGEM}/previsoes_teste_rastreaveis.parquet"
)
if not Path(caminho_previsoes).exists():
    raise FileNotFoundError(
        f"Artefato obrigatório não encontrado: {caminho_previsoes}"
    )
previsoes = pd.read_parquet(caminho_previsoes)

print("Relatórios carregados e validados com sucesso.")


## 3. Desempenho final com incerteza

As estimativas são acompanhadas pelos intervalos de confiança de 95% obtidos por bootstrap municipal. Assim, o gráfico comunica desempenho e incerteza sem tratar alunos do mesmo município como observações territoriais independentes.


In [0]:
# Objetivo: comparar as métricas finais e seus intervalos de confiança.

coluna_metrica = intervalos.columns[0]
dados_metricas = intervalos.rename(columns={coluna_metrica: "metrica"}).copy()

rotulos = {
    "recall_nao_alfabetizado": "Recall classe 0",
    "precision_nao_alfabetizado": "Precisão classe 0",
    "f1_nao_alfabetizado": "F1 classe 0",
    "balanced_accuracy": "Acurácia balanceada",
    "pr_auc_risco": "PR-AUC classe 0",
    "roc_auc_risco": "ROC-AUC classe 0"
}
dados_metricas["rotulo"] = dados_metricas["metrica"].map(rotulos)
dados_metricas = dados_metricas.dropna(subset=["rotulo"])

valores = dados_metricas["estimativa_teste"].astype(float)
erro_inf = valores - dados_metricas["ic_95_inferior"].astype(float)
erro_sup = dados_metricas["ic_95_superior"].astype(float) - valores

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(dados_metricas["rotulo"], valores, color=CORES["azul"], alpha=0.88)
ax.errorbar(
    valores,
    dados_metricas["rotulo"],
    xerr=np.vstack([erro_inf, erro_sup]),
    fmt="none",
    ecolor="#222222",
    capsize=4
)
ax.set_xlim(0, 1)
ax.set_xlabel("Valor da métrica")
ax.set_ylabel("")
ax.set_title("Desempenho final do modelo — teste com IC de 95%")
for indice, valor in enumerate(valores):
    ax.text(valor + 0.012, indice, f"{valor:.3f}", va="center")
fig.tight_layout()
salvar_figura(fig, "01_metricas_finais_ic95.png", "Métricas finais e IC de 95%", "Notebook 08")


## 4. Matriz de confusão

A classe de interesse é `0 = não alfabetizado`. O gráfico destaca acertos e erros operacionais, principalmente os falsos negativos: alunos não alfabetizados que não receberam o alerta do modelo.


In [0]:
# Objetivo: apresentar a matriz de confusão final com contagens e percentuais.

matriz = confusion_matrix(
    previsoes["y_real"],
    previsoes["y_pred"],
    labels=[0, 1]
)
percentuais = matriz / matriz.sum()
anotacoes = np.empty_like(matriz, dtype=object)
for i in range(2):
    for j in range(2):
        anotacoes[i, j] = f"{matriz[i, j]:,}\n{percentuais[i, j]:.1%}".replace(",", ".")

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    matriz,
    annot=anotacoes,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["Previsto: não alfabetizado", "Previsto: alfabetizado"],
    yticklabels=["Real: não alfabetizado", "Real: alfabetizado"],
    ax=ax
)
ax.set_title("Matriz de confusão — conjunto de teste")
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
salvar_figura(fig, "02_matriz_confusao_teste.png", "Matriz de confusão do teste", "Notebook 08")


## 5. Curvas de discriminação

As curvas PR e ROC utilizam a probabilidade de não alfabetização. A PR-AUC é a referência principal porque evidencia melhor a capacidade de identificar a classe de risco diante do desbalanceamento.


In [0]:
# Objetivo: consolidar as curvas Precision-Recall e ROC do modelo final.

y_risco = previsoes["y_real"].eq(0).astype("int8")
proba_risco = previsoes["probabilidade_nao_alfabetizado"].astype(float)
precisao, recall, _ = precision_recall_curve(y_risco, proba_risco)
fpr, tpr, _ = roc_curve(y_risco, proba_risco)
pr_auc = average_precision_score(y_risco, proba_risco)
roc_auc = roc_auc_score(y_risco, proba_risco)
prevalencia = y_risco.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].plot(recall, precisao, color=CORES["vermelho"], linewidth=2, label=f"PR-AUC = {pr_auc:.3f}")
axes[0].axhline(prevalencia, color=CORES["cinza"], linestyle="--", label=f"Prevalência = {prevalencia:.3f}")
axes[0].set(xlabel="Recall", ylabel="Precisão", title="Curva Precision-Recall — classe 0", xlim=(0, 1), ylim=(0, 1))
axes[0].legend()

axes[1].plot(fpr, tpr, color=CORES["azul"], linewidth=2, label=f"ROC-AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], color=CORES["cinza"], linestyle="--", label="Referência aleatória")
axes[1].set(xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos", title="Curva ROC — classe 0", xlim=(0, 1), ylim=(0, 1))
axes[1].legend()

fig.suptitle("Capacidade de discriminação do modelo no teste", y=1.03, fontsize=15, fontweight="bold")
fig.tight_layout()
salvar_figura(fig, "03_curvas_pr_roc_teste.png", "Curvas PR e ROC do teste", "Notebook 08")


## 6. Limiar de decisão

O gráfico registra o compromisso entre recall e precisão observado na validação. A linha vertical identifica o limiar congelado, sem usar o conjunto de teste para reajuste.


In [0]:
# Objetivo: representar recall, precisão e F1 ao longo dos limiares avaliados.

colunas_limiar = {
    "limiar", "recall_nao_alfabetizado",
    "precision_nao_alfabetizado", "f1_nao_alfabetizado"
}
faltantes = colunas_limiar - set(limiares.columns)
if faltantes:
    raise ValueError(f"Colunas ausentes na análise de limiares: {sorted(faltantes)}")

linha_escolhida = limiares.loc[
    limiares["recall_nao_alfabetizado"] >= 0.70
].sort_values("precision_nao_alfabetizado", ascending=False).head(1)
limiar_final = float(linha_escolhida["limiar"].iloc[0]) if len(linha_escolhida) else 0.5

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(limiares["limiar"], limiares["recall_nao_alfabetizado"], label="Recall classe 0", color=CORES["vermelho"], linewidth=2)
ax.plot(limiares["limiar"], limiares["precision_nao_alfabetizado"], label="Precisão classe 0", color=CORES["azul"], linewidth=2)
ax.plot(limiares["limiar"], limiares["f1_nao_alfabetizado"], label="F1 classe 0", color=CORES["verde"], linewidth=2)
ax.axhline(0.70, color=CORES["cinza"], linestyle="--", label="Recall mínimo de desenvolvimento")
ax.axvline(limiar_final, color="#222222", linestyle=":", linewidth=2, label=f"Limiar congelado = {limiar_final:.2f}")
ax.set(xlabel="Limiar", ylabel="Métrica", title="Trade-off do limiar — conjunto de validação", ylim=(0, 1))
ax.legend(loc="best")
fig.tight_layout()
salvar_figura(fig, "04_tradeoff_limiar_validacao.png", "Trade-off do limiar na validação", "Notebook 08")


## 7. Variáveis mais influentes

A importância por permutação mede a perda de PR-AUC quando cada feature é embaralhada. Ela é global, considera a pipeline completa e indica associação preditiva, não causalidade.


In [0]:
# Objetivo: apresentar as quinze maiores importâncias por permutação.

top_importancias = importancias.nlargest(15, "importancia_media").sort_values("importancia_media")
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(
    top_importancias["feature"],
    top_importancias["importancia_media"],
    xerr=top_importancias["desvio_importancia"],
    color=CORES["azul"],
    alpha=0.88,
    capsize=3
)
ax.axvline(0, color="#222222", linewidth=0.8)
ax.set_title("Variáveis mais influentes — importância por permutação")
ax.set_xlabel("Redução média da PR-AUC após permutação")
ax.set_ylabel("")
fig.tight_layout()
salvar_figura(fig, "05_importancia_permutacao_top15.png", "Top 15 variáveis por importância", "Notebook 08")


## 8. Estabilidade regional

A comparação territorial evidencia que uma boa métrica global não garante desempenho homogêneo. Esse gráfico deve ser apresentado junto da limitação registrada no diagnóstico de generalização.


In [0]:
# Objetivo: comparar recall, precisão e F1 da classe 0 entre regiões.

regiao_long = metricas_regiao.melt(
    id_vars=["regiao"],
    value_vars=["recall_nao_alfabetizado", "precision_nao_alfabetizado", "f1_nao_alfabetizado"],
    var_name="metrica",
    value_name="valor"
)
regiao_long["metrica"] = regiao_long["metrica"].map({
    "recall_nao_alfabetizado": "Recall",
    "precision_nao_alfabetizado": "Precisão",
    "f1_nao_alfabetizado": "F1"
})

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=regiao_long, x="regiao", y="valor", hue="metrica", palette=[CORES["vermelho"], CORES["azul"], CORES["verde"]], ax=ax)
ax.axhline(0.70, color=CORES["cinza"], linestyle="--", linewidth=1, label="Recall mínimo de desenvolvimento")
ax.set(title="Desempenho da classe 0 por região", xlabel="", ylabel="Valor da métrica", ylim=(0, 1))
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Métrica", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
salvar_figura(fig, "06_metricas_por_regiao.png", "Desempenho por região", "Notebook 08")


## 9. Prioridade municipal e risco de meta

Os rankings utilizam somente municípios elegíveis do conjunto de teste. Portanto, representam uma demonstração fora da amostra, e não um ranking nacional definitivo.


In [0]:
# Objetivo: consolidar os vinte municípios com maior risco previsto.

top_risco = ranking_risco.nlargest(20, "probabilidade_risco_media").sort_values("probabilidade_risco_media")
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top_risco["rotulo_municipio"], 100 * top_risco["probabilidade_risco_media"], color=CORES["vermelho"])
ax.set(title="Municípios do teste com maior risco médio previsto", xlabel="Probabilidade média de não alfabetização (%)", ylabel="")
fig.tight_layout()
salvar_figura(fig, "07_top20_risco_municipal.png", "Top 20 municípios por risco previsto", "Notebook 09")


In [0]:
# Objetivo: consolidar os vinte menores gaps previstos para a meta.

top_gap = risco_meta.nsmallest(20, "gap_previsto_meta_pp").sort_values("gap_previsto_meta_pp", ascending=False)
cores_gap = np.where(top_gap["gap_previsto_meta_pp"] < 0, CORES["vermelho"], CORES["verde"])
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top_gap["rotulo_municipio"], top_gap["gap_previsto_meta_pp"], color=cores_gap)
ax.axvline(0, color="#222222", linewidth=1)
ax.set(title="Municípios do teste com menor gap previsto para a meta", xlabel="Gap previsto para a meta (pontos percentuais)", ylabel="")
fig.tight_layout()
salvar_figura(fig, "08_top20_gap_meta.png", "Top 20 menores gaps previstos para a meta", "Notebook 09")


## 10. Perfis municipais semelhantes

O silhouette score documenta a escolha do número de clusters. A composição regional mostra como os perfis municipais se distribuem pelo território sem afirmar causalidade geográfica.


In [0]:
# Objetivo: registrar o critério de escolha do número de clusters.

melhor = avaliacao_clusters.loc[avaliacao_clusters["silhouette"].idxmax()]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(avaliacao_clusters["k"], avaliacao_clusters["silhouette"], marker="o", color=CORES["azul"], linewidth=2)
ax.scatter([melhor["k"]], [melhor["silhouette"]], color=CORES["vermelho"], s=110, zorder=3, label=f"Melhor k = {int(melhor['k'])}")
ax.set(title="Seleção do número de clusters municipais", xlabel="Número de clusters (k)", ylabel="Silhouette score")
ax.set_xticks(avaliacao_clusters["k"])
ax.legend()
fig.tight_layout()
salvar_figura(fig, "09_selecao_numero_clusters.png", "Seleção do número de clusters", "Notebook 09")


In [0]:
# Objetivo: consolidar a composição dos clusters por região.

tabela_clusters = clusters_regiao.pivot(index="regiao", columns="cluster", values="percentual_na_regiao").fillna(0).sort_index()
fig, ax = plt.subplots(figsize=(11, 6))
tabela_clusters.plot(kind="bar", stacked=True, colormap="tab10", ax=ax)
ax.set(title="Composição dos clusters por região", xlabel="", ylabel="Percentual dos municípios da região")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"))
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
salvar_figura(fig, "10_clusters_por_regiao.png", "Composição dos clusters por região", "Notebook 09")


## 11. Projeção PCA e manifesto final

A projeção PCA depende das coordenadas calculadas durante a execução do notebook 09. Como a figura já foi validada e salva naquela etapa, ela será copiada sem recomputar os clusters. Em seguida, o manifesto confirma a existência e o tamanho de todas as imagens finais.


In [0]:
# Objetivo: copiar a projeção PCA já produzida e validar o pacote visual.

origem_pca = f"{IMAGENS_APLICACAO}/clusters_municipais_pca.png"
destino_pca = f"{IMAGENS_FINAIS}/11_clusters_municipais_pca.png"

if not Path(origem_pca).exists():
    raise FileNotFoundError(
        "A projeção PCA do notebook 09 não foi encontrada: "
        f"{origem_pca}"
    )

shutil.copy2(origem_pca, destino_pca)
arquivos_gerados.append({
    "arquivo": "11_clusters_municipais_pca.png",
    "titulo": "Projeção PCA dos clusters municipais",
    "origem": "Notebook 09",
    "caminho": destino_pca
})

manifesto_imagens = pd.DataFrame(arquivos_gerados)
manifesto_imagens["existe"] = manifesto_imagens["caminho"].map(lambda caminho: Path(caminho).exists())
manifesto_imagens["tamanho_bytes"] = manifesto_imagens["caminho"].map(lambda caminho: Path(caminho).stat().st_size if Path(caminho).exists() else 0)

if not manifesto_imagens["existe"].all():
    faltantes = manifesto_imagens.loc[~manifesto_imagens["existe"], "arquivo"].tolist()
    raise RuntimeError(f"Imagens finais ausentes: {faltantes}")

if manifesto_imagens["tamanho_bytes"].le(0).any():
    raise RuntimeError("Uma ou mais imagens finais estão vazias.")

manifesto_imagens.to_csv(
    f"{RELATORIO_VISUAL}/manifesto_imagens_finais.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

display(manifesto_imagens)
print(f"Consolidação concluída: {len(manifesto_imagens)} imagens finais.")
print("Diretório:", IMAGENS_FINAIS)


## 12. Resultado esperado

Ao final, a pasta oficial deverá conter 11 arquivos numerados:

1. métricas finais com intervalos de confiança;
2. matriz de confusão;
3. curvas PR e ROC;
4. trade-off do limiar;
5. importância por permutação;
6. desempenho por região;
7. ranking municipal de risco;
8. gap previsto para a meta;
9. seleção do número de clusters;
10. composição dos clusters por região;
11. projeção PCA dos clusters municipais.

Essa seleção cobre desempenho, incerteza, decisão operacional, interpretabilidade, estabilidade territorial e aplicação estratégica.
